### Imports

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_core.documents import Document
from llama_index.core import SimpleDirectoryReader
import asyncio
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk import sent_tokenize

load_dotenv()

c:\Users\kshit\OneDrive\Desktop\CodingNinjasAICourse\Rag\Vector Databases\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

### Config

In [33]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_KEY")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

index_name = "askly-index"

doc_path = Path('docs')
doc_path
nltk_download_path = os.path.join(os.getcwd(), "nltk_data")
nltk.data.path.append(nltk_download_path)

### Setup

In [40]:
nltk.download("punkt_tab", download_dir=nltk_download_path)
model = SentenceTransformer(EMBED_MODEL)
vectorizer = TfidfVectorizer()

def convert_to_readable_sparse_embedding(row):
    coo = row.tocoo()
    return {
        "indices": coo.col.tolist(),
        "values": coo.data.astype(float).tolist(),
    }

pc = Pinecone(api_key=pinecone_api_key)


if not pc.has_index(index_name):
    index = pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

[nltk_data] Downloading package punkt_tab to c:\Users\kshit\OneDrive\D
[nltk_data]     esktop\CodingNinjasAICourse\Rag\Vector
[nltk_data]     Databases\practice_project\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Reading and Chunking of Documents

In [11]:
# load a random file from the docs and apply a regex chunking on it
def load_file(path: Path):
    print(f"Loading file from path: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content

In [ ]:
sample_path = os.path.join(doc_path, 'company_operations.md')
print(Path(sample_path).exists())
file_content = load_file(Path(sample_path))

import re
heading_regex = r'^(#{1,6}\s+.+)$'
matches = re.finditer(heading_regex, file_content, re.MULTILINE)
for match in matches:
    print(match.group())

In [ ]:
markdown_files = SimpleDirectoryReader(input_dir=doc_path).load_data()
markdown_files[0].to_dict()

In [13]:
def chunk_document(text, filename, doc_type,
                   max_tokens=400, overlap_tokens=80,
                   heading_regex=r'^(#{1,6}\s+.+)$',
                   tokenizer=None):
    # tokenizer should provide tokenization/count (e.g., tiktoken or simple word split)
    chunks = []
    # 1) split by headings (simple regex)
    import re
    sections = []
    last_idx = 0
    for m in re.finditer(heading_regex, text, flags=re.MULTILINE):
        if m.start() > last_idx:
            sections.append(text[last_idx:m.start()])
        last_idx = m.start()
    sections.append(text[last_idx:])
    # 2) sub-chunk each section by sentences/tokens
    for s_idx, sec in enumerate(sections):
        sentences = sent_tokenize(sec)
        current = []
        cur_tokens = 0
        chunk_i = 0
        for sent in sentences:
            toks = len((tokenizer(sent) if tokenizer else sent.split()))
            if cur_tokens + toks > max_tokens and current:
                chunk_text = ' '.join(current)
                metadata = {
                    'filename': filename,
                    'doc_type': doc_type,
                    'section_index': s_idx,
                }
                chunk = Document(page_content=chunk_text, metadata=metadata)
                chunks.append(chunk)
                # prepare overlap
                if overlap_tokens > 0:
                    # keep last sentences that approx overlap
                    overlap = []
                    o_tokens = 0
                    for st in reversed(current):
                        ot = len((tokenizer(st) if tokenizer else st.split()))
                        if o_tokens + ot > overlap_tokens:
                            break
                        overlap.insert(0, st); o_tokens += ot
                    current = overlap
                    cur_tokens = o_tokens
                else:
                    current = []; cur_tokens = 0
                chunk_i += 1
            current.append(sent); cur_tokens += toks
        if current:
            metadata = {
                'filename': filename,
                'doc_type': doc_type,
                'section_index': s_idx,
            }
            chunk = Document(page_content=' '.join(current), metadata=metadata)
            chunks.append(chunk)
    return chunks

### Trying ChatOllama

In [ ]:
# from langchain_ollama import ChatOllama

# ollama_llm = ChatOllama(
#     model="mistral:latest",
#     temperature=0,
# )
# messages = [
#     (
#         "system",
#         "You are a helpful assistant",
#     ),
#     ("human", "What is the use of the solar system?"),
# ]
# ai_msg = ollama_llm.invoke(messages)
# ai_msg.content

### Creating dense and sparse embeddings

In [31]:
sample_content = load_file(Path(sample_path))
sample_chunks = chunk_document(filename="company_operations.md", doc_type="markdown", text=sample_content)

Loading file from path: docs\company_operations.md


In [21]:
def build_chunks():
    final_chunks = []
    # loop through the docs directory and create chunks
    for file in doc_path.iterdir():
        content = load_file(file)
        chunks = chunk_document(filename=file.name, doc_type=file.suffix.lstrip('.'), text=content)
        final_chunks.extend(chunks)
    return final_chunks

In [22]:
def create_vector_to_upload():
    vectors = []
    chunks = build_chunks()
    text_corpus = [chunk.page_content for chunk in chunks]
    dense_embs = model.encode(text_corpus, convert_to_numpy=True, normalize_embeddings=True, batch_size=16)
    sparse_embeddings = vectorizer.fit_transform(text_corpus)

    for i, chunk in enumerate(chunks):
        sparse_emb_row = sparse_embeddings[i]
        readable_sparse_emb = convert_to_readable_sparse_embedding(sparse_emb_row)
        meta = {
            **chunk.metadata,
            'content': chunk.page_content[:1200]
        }
        vector = {
            'id': f"{chunk.metadata['filename']}_sec{chunk.metadata['section_index']}",
            'values': dense_embs[i].tolist(),
            'sparse_values': readable_sparse_emb,
            'metadata': meta
        }
        vectors.append(vector)
    return vectors

In [24]:
final_vectors = create_vector_to_upload()

Loading file from path: docs\company_operations.md
Loading file from path: docs\customer_service_guide.md
Loading file from path: docs\finance_procedures.md
Loading file from path: docs\hr_handbook.md
Loading file from path: docs\it_support_guide.md
Loading file from path: docs\sample_policy.md
Loading file from path: docs\technical_documentation.md


In [41]:

index.upsert(vectors=final_vectors)

{'upserted_count': 542}

## Retriver Module

In [44]:
query = "What is the company's policy on remote work?"

query_vector = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
results = index.query(
    vector=query_vector.tolist(),
    top_k=5,
    include_metadata=True,
    include_values=False,
    # include_sparse_values=True,
)
[res['metadata']['content'] for res in results['matches']]

['### Remote Work & Hybrid Operations\n\n**Hybrid Work Policy**\n- **Standard Model**: 3 days in office, 2 days remote\n- **Full Remote**: Available with VP approval and quarterly review\n- **Home Office Setup**: $2,000 stipend for ergonomic equipment\n- **Internet Reimbursement**: $100/month for high-speed internet\n- **Co-working Spaces**: $300/month allowance for approved spaces\n\n**Remote Work Requirements**\n- **Workspace**: Private, professional environment suitable for video calls\n- **Equipment**: Company-provided laptop, monitor, peripherals\n- **Security**: VPN required, secure WiFi, no public networks\n- **Availability**: Must be available during core hours\n- **Travel**: Occasional travel to office locations as required\n\n---',
 '## Workplace Standards and Remote Work\n\nStandard Hours: Orion’s standard workweek is Monday–Friday, 40 hours. Typical core hours are 10:00–16:00 local, with flexibility to start between 7:00–10:00 and finish between 16:00–19:00, subject to clie